In [0]:
dbutils.widgets.text('SourceSystem', '')
dbutils.widgets.text('TableName', '')
dbutils.widgets.text('LoadType', '')
dbutils.widgets.text('DWRunID', '')

In [0]:
source_system = dbutils.widgets.get('SourceSystem')
table_name = dbutils.widgets.get('TableName')
load_type = dbutils.widgets.get('LoadType')
dw_run_id = dbutils.widgets.get('DWRunID')

In [0]:
%run "../Library/DWFunctions"

In [0]:
df_silver_configurations = read_delta_table('metadata.silver.silver_metadata_configurations')

In [0]:
silver_config = df_silver_configurations.collect()[0].asDict()

### Converting Spark DataFrame to Dictionary

The pattern `df.collect()[0].asDict()` converts a Spark DataFrame row to a Python dictionary:

1. **`.collect()`** - Spark action that brings all rows from the distributed cluster to the driver as a list of Row objects
   - Returns: `[Row(...), Row(...), ...]`
   - ⚠️ Only use on small DataFrames (loads entire dataset into driver memory)

2. **`[0]`** - Selects the first Row object from the list
   - Returns: `Row(SourceSystem='kaggle', BronzeTablePath='...', ...)`

3. **`.asDict()`** - Converts the Row object to a standard Python dictionary
   - Returns: `{'SourceSystem': 'kaggle', 'BronzeTablePath': '...', ...}`

**Alternative for single-row DataFrames:**
```python
silver_config = df_silver_configurations.first().asDict()
```
`.first()` is more efficient than `.collect()[0]` when you only need one row.

In [0]:
silver_columns = silver_config.get('SilverColumns')
bronze_table_path = silver_config.get('BronzeTablePath')

In [0]:
df_bronze = read_delta_table(bronze_table_path)

### Using selectExpr to Rename and Select Columns

The code below uses `selectExpr` to efficiently rename and select columns from the `df_bronze` DataFrame based on the mappings provided in `silver_columns`.

- **`selectExpr`**: Allows SQL-like expressions for column selection and renaming in Spark DataFrames.
- **Column Mapping**: For each `(source, target)` pair in `silver_columns`, an expression `"source as target"` is generated.
- **Unpacking with `*`**: The `*` operator unpacks the list of expressions so each becomes a separate argument to `selectExpr`.

This creates a new DataFrame (`df_silver`) with columns renamed and selected as specified.

python
df_silver = df_bronze.selectExpr(*[f"{source} as {target}" for source, target in silver_columns])

In [0]:
df_silver = df_bronze.selectExpr(*[f"{source} as {target}" for source, target in silver_columns])

In [0]:
%sql
USE CATALOG silver;
CREATE SCHEMA kaggle;

In [0]:
write_dataframe_to_delta_table(df_silver, f'silver.{source_system}.{table_name}')